# Hyper-Knowledge · Python / Notebook

一套核心：构图 → 校验 → 查看成员角色 → 嵌入工作台。
No model service is needed for this structured-data tutorial.

Install in the **current kernel** when needed (run in a separate code cell):
```python
%pip install "hyper-knowledge[notebook] @ git+https://github.com/hanxiangmin/Hyper-Knowledge.git"
```
Restart the kernel after installing or upgrading the runtime. Select the kernel for the Python/Conda environment where you installed it.


In [ ]:
import sys
from pathlib import Path
from tempfile import mkdtemp
from hyperknowledge import import_graph, read_bundle, validate_bundle, render_bundle_html
print(sys.executable)  # Check the selected kernel environment.


In [ ]:
# Explicit tutorial assertions, not a model extraction result.
graph = {
  "nodes": [
    {
      "id": "su",
      "label": "苏轼",
      "type": "person"
    },
    {
      "id": "zhe",
      "label": "苏辙",
      "type": "person"
    },
    {
      "id": "year",
      "label": "1101年",
      "type": "time"
    },
    {
      "id": "place",
      "label": "常州",
      "type": "place"
    }
  ],
  "assertions": [
    {
      "id": "return",
      "predicate": "北归",
      "epistemic_status": "example_assertion",
      "evidence_refs": [
        "return-span"
      ]
    },
    {
      "id": "brothers",
      "predicate": "兄弟",
      "epistemic_status": "example_assertion",
      "evidence_refs": [
        "brothers-span"
      ]
    }
  ],
  "members": [
    {
      "assertion_id": "return",
      "node_id": "su",
      "role": "北归者"
    },
    {
      "assertion_id": "return",
      "node_id": "year",
      "role": "时间"
    },
    {
      "assertion_id": "return",
      "node_id": "place",
      "role": "到达地"
    },
    {
      "assertion_id": "brothers",
      "node_id": "su",
      "role": "兄长"
    },
    {
      "assertion_id": "brothers",
      "node_id": "zhe",
      "role": "弟弟"
    }
  ],
  "evidence": [
    {
      "id": "return-span",
      "type": "source_text_span",
      "source": "notes.md",
      "line_start": 2,
      "line_end": 2,
      "quote": "1101年，苏轼北归抵达常州。"
    },
    {
      "id": "brothers-span",
      "type": "source_text_span",
      "source": "notes.md",
      "line_start": 3,
      "line_end": 3,
      "quote": "苏轼与苏辙是兄弟，苏轼是兄长，苏辙是弟弟。"
    }
  ]
}


In [ ]:
# Keep HTML beneath the notebook directory so Jupyter can serve the iframe.
output = Path(mkdtemp(prefix="hk-notebook-", dir=Path.cwd()))
source = output / "notes.md"
source.write_text("# 教程样例 / Tutorial fixture\n1101年，苏轼北归抵达常州。\n苏轼与苏辙是兄弟，苏轼是兄长，苏辙是弟弟。\n\n这份短文仅用于演示数据流，并非独立核验的历史资料。\n", encoding="utf-8")
result = import_graph(graph, sources={"notes.md": source}, output_dir=output / "bundle", quality="showcase")
result.to_dict()


In [ ]:
tables = read_bundle(result.bundle_path)
[(node["label"], node["type"]) for node in tables["nodes"]]


In [ ]:
[(edge["predicate"], [(m["node_id"], m["role"]) for m in tables["members"] if m["assertion_id"] == edge["id"]]) for edge in tables["assertions"]]


In [ ]:
validation = validate_bundle(result.bundle_path, quality="showcase")
assert validation["status"] == "passed"
assert result.node_count == 4 and result.hyperedge_count == 2
validation["status"]


In [ ]:
from IPython.display import IFrame, display
page = output / "workbench.html"
render_bundle_html(result.bundle_path, page)
display(IFrame(page.relative_to(Path.cwd()).as_posix(), width="100%", height=720))


## Parse your own document / 解析自己的文档
This separate route uses a model service. Configure `hk config llm`, or explicitly pass a LangChain-compatible chat client.
Embeddings are needed only if you request indexing.

```python
from hyperknowledge import extract_file, render_bundle_html
result = extract_file("notes.md", output_dir="output/extracted", language="zh")
render_bundle_html(result.bundle_path, "output/extracted/workbench.html")
```
